In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
# ── 1. CIFAR-10 with augmentation ─────────────────
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465),
                         (0.2470, 0.2435, 0.2616))
])
test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465),
                         (0.2470, 0.2435, 0.2616))
])

train_ds = datasets.CIFAR10('./data', train=True,  download=True, transform=train_transform)
test_ds  = datasets.CIFAR10('./data', train=False, download=True, transform=test_transform)
train_loader = DataLoader(train_ds, batch_size=128, shuffle=True)
test_loader  = DataLoader(test_ds,  batch_size=128, shuffle=False)


 14%|█████████▎                                                           | 23.1M/170M [01:24<04:34, 537kB/s]

In [ ]:
# ── 2. Baseline CNN (no BN, no Dropout) ───────────
class BaselineCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Flatten(),
            nn.Linear(128*4*4, 256), nn.ReLU(),
            nn.Linear(256, 10)
        )
    def forward(self, x): return self.net(x)


In [ ]:
# ── 3. Regularized CNN (BN + Dropout) ─────────────
class RegularizedCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128*4*4, 256),
            nn.BatchNorm1d(256), nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, 10)
        )
    def forward(self, x): return self.classifier(self.features(x))


In [ ]:
# ── 4. Training helpers ────────────────────────────
def train_epoch(model, loader, optimizer, criterion):
    model.train()
    loss_sum, correct = 0, 0
    for X, y in loader:
        X, y = X.to(device), y.to(device)
        optimizer.zero_grad()
        out = model(X)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
        loss_sum += loss.item() * len(X)
        correct  += (out.argmax(1) == y).sum().item()
    return loss_sum / len(loader.dataset), correct / len(loader.dataset)

def evaluate(model, loader, criterion):
    model.eval()
    loss_sum, correct = 0, 0
    with torch.no_grad():
        for X, y in loader:
            X, y = X.to(device), y.to(device)
            out = model(X)
            loss = criterion(out, y)
            loss_sum += loss.item() * len(X)
            correct  += (out.argmax(1) == y).sum().item()
    return loss_sum / len(loader.dataset), correct / len(loader.dataset)


In [ ]:
# ── 5. Run ablation study ──────────────────────────
EPOCHS = 10
results = {}

for name, model, wd in [
    ('Baseline',           BaselineCNN(),     0),
    ('BatchNorm+Dropout',  RegularizedCNN(),  1e-4)
]:
    print(f"\n── {name} ──")
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=wd)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
    accs = []
    for epoch in range(1, EPOCHS+1):
        train_epoch(model, train_loader, optimizer, criterion)
        _, te_acc = evaluate(model, test_loader, criterion)
        scheduler.step()
        accs.append(te_acc)
        if epoch % 5 == 0:
            print(f"  Epoch {epoch} | Test Acc: {te_acc:.4f}")
    results[name] = accs

In [ ]:
# ── 6. Plot comparison ─────────────────────────────
plt.figure(figsize=(9, 4))
for name, accs in results.items():
    plt.plot(accs, lw=2, label=name)
plt.title('Baseline vs Regularized CNN — CIFAR-10')
plt.xlabel('Epoch')
plt.ylabel('Test Accuracy')
plt.legend()
plt.tight_layout()
plt.show()

print("\n── Final Results ──")
for name, accs in results.items():
    print(f"{name:25s}: {accs[-1]:.4f}")